# GOLD ATP PLAYER-TOURNAMENTS STATS

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("fact_player_tournament_stats").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
# silver
tb_player_match = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

# gold
tb_tournaments = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

tb_entry = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_entry")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

tb_players = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

## Player-Tournaments

In [17]:
round_order = (
    f.when(f.col("MATCH_ROUND") == "F", 11) # Final
     .when(f.col("MATCH_ROUND") == "SF", 10) # Semi Final
     .when(f.col("MATCH_ROUND") == "BR", 9) # Third Place
     .when(f.col("MATCH_ROUND") == "QF", 8) # Quarte final
     .when(f.col("MATCH_ROUND") == "R16", 7) # Round of 16
     .when(f.col("MATCH_ROUND") == "R32", 6) # Round of 32
     .when(f.col("MATCH_ROUND") == "R64", 5) # Round of 64
     .when(f.col("MATCH_ROUND") == "R128", 4) # Round of 128
     .when(f.col("MATCH_ROUND") == "ER", 3) # Early Round
     .when(f.col("MATCH_ROUND") == "RR", 2) # Round Robin
     .otherwise(1)
)

tourney_stats = (
    tb_player_match
    .groupBy("TOURNEY_ID", "PLAYER_ID")
    .agg(
        f.first("PLAYER_ENTRY").alias("PLAYER_ENTRY"),
        f.coalesce(f.first("PLAYER_SEED").cast('int'), f.lit(-1)).alias("PLAYER_SEED"),
        f.coalesce(f.first("PLAYER_RANK_PTS").cast('int'), f.lit(0)).alias("PLAYER_RANK_PTS"),
        f.coalesce(f.first("PLAYER_RANK").cast('int'), f.lit(0)).alias("PLAYER_RANK"),

        f.max(
            f.when((f.col("MATCH_ROUND") == "F") & (f.col("PLAYER_IS_WINNER") == True), 1).otherwise(0)
        ).alias("IS_CHAMPION"),
        
        f.max_by(f.col("MATCH_ROUND"), round_order).alias("LAST_ROUND_PLAYED"),

        f.count("MATCH_ID").alias("TOTAL_MATCHES"),
        f.coalesce(f.sum(f.col("MATCH_DURATION_M").cast("int")), f.lit(0)).alias("TOTAL_MIN_IN_GAME"),
        f.coalesce(f.max(f.col("MATCH_DURATION_M").cast("int")), f.lit(0)).alias("LONGEST_MATCH"),


        f.coalesce(f.sum(f.col("PLAYER_ACES").cast('int')), f.lit(0)).alias("TOTAL_ACES"), 
        f.coalesce(f.sum(f.col("PLAYER_DB_FAULTS").cast('int')), f.lit(0)).alias("TOTAL_DB_FAULTS"), 
        f.coalesce(f.sum(f.col("PLAYER_SERVE_PTS").cast('int')), f.lit(0)).alias("TOTAL_SERVE_PTS"), 
        f.coalesce(f.sum(f.col("PLAYER_1ST_SERVES_IN").cast('int')), f.lit(0)).alias("TOTAL_1ST_SERVES_IN"), 
        f.coalesce(f.sum(f.col("PLAYER_1ST_SERVE_PTS_WON").cast('int')), f.lit(0)).alias("TOTAL_1ST_SERVE_PTS_WON"), 
        f.coalesce(f.sum(f.col("PLAYER_2ND_SERVE_PTS_WON").cast('int')), f.lit(0)).alias("TOTAL_2ND_SERVE_PTS_WON"), 
        f.coalesce(f.sum(f.col("PLAYER_SERVE_GAMES").cast('int')), f.lit(0)).alias("TOTAL_SERVE_GAMES"), 
        f.coalesce(f.sum(f.col("PLAYER_BP_SAVED").cast('int')), f.lit(0)).alias("TOTAL_BP_SAVED"), 
        f.coalesce(f.sum(f.col("PLAYER_BP_FACED").cast('int')), f.lit(0)).alias("TOTAL_BP_FACED") 
    )
)

In [18]:
df = (
    tourney_stats.alias("s")
    .join(tb_tournaments.alias("t"), "TOURNEY_ID", 'left')
    .join(tb_players.alias("p"), "PLAYER_ID", 'left')
    .join(tb_entry.alias("e"), f.expr("s.PLAYER_ENTRY <=> e.ENTRY_TYPE_ID"), 'left')
    
    .select(
        f.col("p.SK_PLAYER"),
        f.col("t.SK_TOURNEY"),
        f.col("e.SK_ENTRY_TYPE"),

        f.col("s.PLAYER_SEED"),
        f.col("s.PLAYER_RANK_PTS"),
        f.col("s.PLAYER_RANK"),
        f.col("s.IS_CHAMPION"),
        f.col("s.LAST_ROUND_PLAYED"),
        f.col("s.TOTAL_MATCHES"),
        f.col("s.TOTAL_MIN_IN_GAME"),
        f.col("s.LONGEST_MATCH"),
        f.col("s.TOTAL_ACES"), 
        f.col("s.TOTAL_DB_FAULTS"), 
        f.col("s.TOTAL_SERVE_PTS"), 
        f.col("s.TOTAL_1ST_SERVES_IN"), 
        f.col("s.TOTAL_1ST_SERVE_PTS_WON"), 
        f.col("s.TOTAL_2ND_SERVE_PTS_WON"), 
        f.col("s.TOTAL_SERVE_GAMES"), 
        f.col("s.TOTAL_BP_SAVED"), 
        f.col("s.TOTAL_BP_FACED") 
    )
    .distinct()
)

## Validade final dataframe

In [19]:
if tourney_stats.count() == df.count():
    print('ok')
else: 
    raise

ok


## Save dataframe

### Local

In [20]:
df.toPandas().to_csv(
    r"../../../data/gold/fact/fact_player_tournament_stats.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.fact_player_tournament_stats")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)